Data Skew in PySpark
Data skew occurs when data is unevenly distributed across partitions, causing some partitions to have significantly more data than others.


Why It's a Problem

Partition 1: ████████████████████  500k rows  ← takes 10 mins

Partition 2: ███                    30k rows  ← done in 1 min

Partition 3: ███                    20k rows  ← done in 1 min

Partition 4: ██                     10k rows  ← done in 30 secs

The entire job waits for Partition 1 to finish — wasting cluster resources.

Common Causes
python
 1. Joining on a column with many NULLs or one dominant value

df.join(df2, on='DeptName')   # if 80% of rows are "Sales" → skew

 2. GroupBy on low-cardinality column

df.groupBy('Country').agg(sum('Salary'))  # if 90% rows are "India"

 3. Poor partitioning
 
df.repartition(10)  # data may not split evenly

In [0]:
#Check partition sizes
df.rdd.mapPartitionsWithIndex(
    lambda i, it: [(i, sum(1 for _ in it))]
).toDF(['partition', 'count']).show()

#Or in Spark UI → Stages tab → look for tasks with very different durations

In [0]:
#Check number of partitions
#Does not work with Serverless compute
df.rdd.getNumPartitions()

In [0]:
# Alt for getNumPartitions for serverless compute
# People dataset — good for skew practice
df = spark.read.csv('/databricks-datasets/adult/adult.data',
                    header=False, inferSchema=True)

display(df)
# ✅ Serverless compatible
print("Number of partitions:", len(df.inputFiles())) 

In [0]:
# Check workspace catalog (most likely to have write access)
spark.sql("SHOW SCHEMAS IN workspace").display()

# Replace 'your_schema' with result from Step 1
spark.sql("SHOW VOLUMES IN workspace.information_schema").display()

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.my_schema").display()
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.my_schema.my_volume").display()

In [0]:
df = df.repartition(10)

df.write.mode('overwrite').parquet('/Volumes/workspace/my_schema/my_volume/partition_check/')

part_count = len([f for f in dbutils.fs.ls('/Volumes/workspace/my_schema/my_volume/partition_check/') 
                  if f.name.startswith('part-')])

print("Number of partitions:", part_count)  # ✅ Should print 10

In [0]:
from pyspark.sql.functions import spark_partition_id

df1 = df.select(spark_partition_id().alias('partid')).groupBy('partid').count()
display(df1)